# Flickr8k Image-Text ImageBind-Style Mini

This Kaggle notebook adapts the existing ESC-50 ImageBind-style idea from audio-text to image-text on Flickr8k.

Goal: train a small model from scratch for at least one epoch using `(image, caption)` positive pairs, learn a joint embedding space with symmetric InfoNCE contrastive loss, and evaluate image-to-text plus text-to-image retrieval.

This notebook does not use pretrained CLIP, OpenCLIP, ImageBind, or audio. The image encoder is a randomly initialized ResNet18 and the text encoder is a small trainable embedding + GRU network.

## 1. Imports and Config

The config defaults are Kaggle-friendly. `FORCE_CPU=False` allows GPU use when CUDA is actually compatible; the next cell tests CUDA with a small matrix multiplication and falls back to CPU if needed.

In [ ]:
import json
import math
import os
import random
import re
import string
import textwrap
import time
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import torchvision
from torchvision import transforms
from torchvision.models import resnet18

try:
    from sklearn.metrics import pairwise_distances
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

@dataclass
class Config:
    DATA_ROOT: str = "/kaggle/input/flickr8k"
    IMAGE_SIZE: int = 224
    EMBED_DIM: int = 256
    BATCH_SIZE: int = 32
    EPOCHS: int = 1
    LR: float = 3e-4
    WEIGHT_DECAY: float = 1e-4
    TEMPERATURE: float = 0.07
    NUM_WORKERS: int = 2
    FORCE_CPU: bool = False
    MAX_LEN: int = 32
    OUTPUT_DIR: str = "outputs"
    CHECKPOINT_DIR: str = "checkpoints"
    ABLATION_MAX_BATCHES: int = 100

CFG = Config()
OUTPUT_DIR = Path(CFG.OUTPUT_DIR)
CHECKPOINT_DIR = Path(CFG.CHECKPOINT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(asdict(CFG))
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)

## 2. Device Setup With CUDA Compatibility Test

Some Kaggle images can expose a CUDA device that is incompatible with the installed PyTorch build. This cell tests CUDA with a small matrix multiplication first. If it catches a CUDA runtime error such as `no kernel image is available for execution on the device`, training continues on CPU.

In [ ]:
def setup_device(force_cpu=False):
    if force_cpu:
        print("FORCE_CPU=True, using CPU.")
        return torch.device("cpu")
    if not torch.cuda.is_available():
        print("CUDA is not available, using CPU.")
        return torch.device("cpu")

    try:
        device = torch.device("cuda")
        a = torch.randn(64, 64, device=device)
        b = torch.randn(64, 64, device=device)
        c = a @ b
        torch.cuda.synchronize()
        print("CUDA test passed:", torch.cuda.get_device_name(0))
        return device
    except RuntimeError as exc:
        msg = str(exc)
        print("CUDA test failed, falling back to CPU.")
        print("RuntimeError:", msg[:500])
        if "no kernel image is available for execution on the device" in msg:
            print("Detected CUDA architecture mismatch: no kernel image is available for execution on the device.")
        return torch.device("cpu")

DEVICE = setup_device(CFG.FORCE_CPU)
print("Device:", DEVICE)

## 3. Flickr8k Path Detection and Caption Parser

The notebook first checks `/kaggle/input/flickr8k`, then common Kaggle alternatives. It recursively detects `captions.txt`, handles both common caption formats, and locates the image directory by checking real image filenames.

In [ ]:
COMMON_DATA_ROOTS = [
    Path(CFG.DATA_ROOT),
    Path("/kaggle/input/flickr8kimagescaptions"),
    Path("/kaggle/input/flickr-8k"),
]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def candidate_roots():
    roots = []
    for p in COMMON_DATA_ROOTS:
        if p.exists() and p not in roots:
            roots.append(p)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for child in sorted(kaggle_input.iterdir()):
            if child.is_dir() and child not in roots:
                roots.append(child)
    if not roots:
        roots.append(Path.cwd())
    return roots


def find_captions_file():
    for root in candidate_roots():
        direct = root / "captions.txt"
        if direct.exists():
            return direct
        try:
            matches = list(root.rglob("captions.txt"))
        except Exception as exc:
            print(f"Skipping {root}: {exc}")
            matches = []
        if matches:
            return matches[0]
    searched = ", ".join(str(x) for x in candidate_roots())
    raise FileNotFoundError(f"Could not find captions.txt. Searched: {searched}")


def clean_image_name(name):
    name = str(name).strip().strip('"').strip("'")
    name = re.sub(r"#\d+$", "", name)
    return Path(name).name


def parse_caption_line(line):
    line = line.strip()
    if not line:
        return None
    lower = line.lower()
    if lower in {"image,caption", "image_name,comment", "filename,caption"}:
        return None

    # Newer Flickr8k Kaggle format: image,caption
    if "," in line:
        first, rest = line.split(",", 1)
        if Path(clean_image_name(first)).suffix.lower() in IMAGE_EXTS:
            return clean_image_name(first), rest.strip().strip('"')

    # Older format: image_name#0<TAB>caption or image_name#0 caption
    match = re.match(r"^(.+?\.(?:jpg|jpeg|png|bmp|webp))(?:#\d+)?[\t ]+(.*)$", line, flags=re.IGNORECASE)
    if match:
        return clean_image_name(match.group(1)), match.group(2).strip().strip('"')
    return None


def load_captions(captions_path):
    rows = []
    with open(captions_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parsed = parse_caption_line(line)
            if parsed is not None:
                image, caption = parsed
                if image and caption:
                    rows.append({"image_id": image, "caption": caption})

    if not rows:
        # Fallback for quoted CSV files.
        df = pd.read_csv(captions_path)
        cols = {c.lower().strip(): c for c in df.columns}
        image_col = cols.get("image") or cols.get("filename") or cols.get("image_name")
        caption_col = cols.get("caption") or cols.get("comment") or cols.get("caption_text")
        if image_col is None or caption_col is None:
            raise ValueError(f"Could not parse captions file columns: {df.columns.tolist()}")
        rows = [
            {"image_id": clean_image_name(img), "caption": str(cap).strip()}
            for img, cap in zip(df[image_col], df[caption_col])
            if str(img).strip() and str(cap).strip()
        ]

    df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    df = df[df["image_id"].str.lower().str.endswith(tuple(IMAGE_EXTS))].reset_index(drop=True)
    return df


def find_images_dir(root, captions_path, image_ids):
    parents = [captions_path.parent, root, root / "flickr8k", root / "Flickr8k"]
    names = ["Images", "images", "Flicker8k_Dataset", "Flickr8k_Dataset", "flickr8k/Images"]
    candidates = []
    for parent in parents:
        for name in names:
            candidates.append(parent / name)
        candidates.append(parent)
    sample_names = list(dict.fromkeys(image_ids[:50]))
    for cand in candidates:
        if cand.exists() and cand.is_dir():
            hits = sum((cand / name).exists() for name in sample_names)
            if hits > 0:
                return cand

    search_base = root if root.exists() else captions_path.parent
    for name in sample_names[:10]:
        try:
            matches = list(search_base.rglob(name))
        except Exception:
            matches = []
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not locate Flickr8k Images directory from captions filenames.")

CAPTIONS_PATH = find_captions_file()
DATA_ROOT = next((r for r in candidate_roots() if CAPTIONS_PATH.is_relative_to(r)), CAPTIONS_PATH.parent)
all_captions_df = load_captions(CAPTIONS_PATH)
IMAGES_DIR = find_images_dir(DATA_ROOT, CAPTIONS_PATH, all_captions_df["image_id"].tolist())

# Keep only rows whose image file exists.
all_captions_df["image_path"] = all_captions_df["image_id"].apply(lambda x: str(IMAGES_DIR / x))
all_captions_df = all_captions_df[all_captions_df["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

print("DATA_ROOT:", DATA_ROOT)
print("CAPTIONS_PATH:", CAPTIONS_PATH)
print("IMAGES_DIR:", IMAGES_DIR)
print("Caption pairs:", len(all_captions_df))
print("Unique images:", all_captions_df["image_id"].nunique())
display(all_captions_df.head())

## 4. Split by Image Filename

Flickr8k has around five captions per image. Splitting by caption would leak the same image into train and test. This split is done by unique `image_id`: 80% train, 10% validation, 10% test.

In [ ]:
def split_by_image_id(df, seed=42, train_ratio=0.8, val_ratio=0.1):
    image_ids = sorted(df["image_id"].unique().tolist())
    rng = random.Random(seed)
    rng.shuffle(image_ids)
    n = len(image_ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_ids = set(image_ids[:n_train])
    val_ids = set(image_ids[n_train:n_train + n_val])
    test_ids = set(image_ids[n_train + n_val:])

    train_df = df[df["image_id"].isin(train_ids)].reset_index(drop=True)
    val_df = df[df["image_id"].isin(val_ids)].reset_index(drop=True)
    test_df = df[df["image_id"].isin(test_ids)].reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = split_by_image_id(all_captions_df, seed=SEED)
print(f"Train pairs: {len(train_df):,} | images: {train_df['image_id'].nunique():,}")
print(f"Val pairs:   {len(val_df):,} | images: {val_df['image_id'].nunique():,}")
print(f"Test pairs:  {len(test_df):,} | images: {test_df['image_id'].nunique():,}")

assert set(train_df.image_id).isdisjoint(set(val_df.image_id))
assert set(train_df.image_id).isdisjoint(set(test_df.image_id))
assert set(val_df.image_id).isdisjoint(set(test_df.image_id))

## 5. Text Tokenizer

The tokenizer is intentionally simple: lowercase, strip punctuation, build vocabulary from training captions only, and pad/truncate to `max_len=32`.

In [ ]:
class SimpleTokenizer:
    PAD = 0
    UNK = 1

    def __init__(self, texts, max_len=32, min_freq=1):
        self.max_len = max_len
        counts = defaultdict(int)
        for text in texts:
            for tok in self.tokenize(text):
                counts[tok] += 1
        self.stoi = {"<pad>": self.PAD, "<unk>": self.UNK}
        for tok, count in sorted(counts.items()):
            if count >= min_freq and tok not in self.stoi:
                self.stoi[tok] = len(self.stoi)
        self.itos = {i: s for s, i in self.stoi.items()}

    @staticmethod
    def tokenize(text):
        text = str(text).lower()
        text = text.translate(str.maketrans("", "", string.punctuation))
        return text.strip().split()

    def encode(self, text):
        ids = [self.stoi.get(tok, self.UNK) for tok in self.tokenize(text)]
        ids = ids[:self.max_len]
        ids += [self.PAD] * (self.max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)

TOKENIZER = SimpleTokenizer(train_df["caption"].tolist(), max_len=CFG.MAX_LEN)
print("Vocabulary size:", len(TOKENIZER.stoi))
print("Example:", train_df.iloc[0]["caption"])
print("Tokens:", TOKENIZER.encode(train_df.iloc[0]["caption"])[:12].tolist())

## 6. Flickr8k Dataset and DataLoaders

Each row is one positive `(image, caption)` pair. Since each image has multiple captions, the same image can appear in multiple rows with different captions. Evaluation later groups positives by `image_id`.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class Flickr8kDataset(Dataset):
    def __init__(self, dataframe, tokenizer=None, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        caption = str(row["caption"])
        image_id = str(row["image_id"])
        item = {"image": image, "caption": caption, "image_id": image_id}
        if self.tokenizer is not None:
            item["tokens"] = self.tokenizer.encode(caption)
        return item


def collate_fn(batch):
    return {
        "image": torch.stack([x["image"] for x in batch]),
        "tokens": torch.stack([x["tokens"] for x in batch]),
        "caption": [x["caption"] for x in batch],
        "image_id": [x["image_id"] for x in batch],
    }


def make_loader(df, transform, shuffle=False, batch_size=None):
    return DataLoader(
        Flickr8kDataset(df, tokenizer=TOKENIZER, transform=transform),
        batch_size=batch_size or CFG.BATCH_SIZE,
        shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collate_fn,
        drop_last=False,
    )

train_loader = make_loader(train_df, train_transform, shuffle=True)
val_loader = make_loader(val_df, eval_transform, shuffle=False)
test_loader = make_loader(test_df, eval_transform, shuffle=False)

batch = next(iter(train_loader))
print(batch["image"].shape, batch["tokens"].shape, batch["image_id"][0], batch["caption"][0])

## 7. Model: Image Encoder, Text Encoder, Joint Embedding

The image encoder uses `torchvision.models.resnet18(weights=None)`, so it trains from scratch. The text encoder is an embedding layer plus GRU. Both projections are L2-normalized into the same `EMBED_DIM` space.

In [ ]:
def make_resnet18_no_pretrain():
    try:
        return resnet18(weights=None)
    except TypeError:
        return resnet18(pretrained=False)


def looks_like_cuda_runtime_error(exc):
    msg = str(exc).lower()
    return any(key in msg for key in [
        "cuda",
        "cudnn",
        "no kernel image is available for execution on the device",
        "invalid device function",
    ])

class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        backbone = make_resnet18_no_pretrain()
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.proj = nn.Sequential(
            nn.Linear(in_features, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, images):
        features = self.backbone(images)
        return self.proj(features)

class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, token_dim=128, hidden_dim=256, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, token_dim, padding_idx=pad_id)
        self.gru = nn.GRU(token_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, tokens):
        mask = tokens.ne(self.pad_id)
        x = self.embedding(tokens)
        h, _ = self.gru(x)
        h = h * mask.unsqueeze(-1)
        pooled = h.sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp_min(1)
        return self.proj(pooled)

class ImageTextContrastiveModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim)
        self.text_encoder = TextEncoder(vocab_size, embed_dim)

    def encode_image(self, images):
        return F.normalize(self.image_encoder(images), dim=-1)

    def encode_text(self, tokens):
        return F.normalize(self.text_encoder(tokens), dim=-1)

    def forward(self, images, tokens):
        image_emb = self.encode_image(images)
        text_emb = self.encode_text(tokens)
        return image_emb, text_emb


def build_model_with_cuda_smoke_test():
    global DEVICE
    model = ImageTextContrastiveModel(len(TOKENIZER.stoi), CFG.EMBED_DIM).to(DEVICE)
    if DEVICE.type == "cuda":
        try:
            model.eval()
            with torch.no_grad():
                _ = model(batch["image"][:2].to(DEVICE), batch["tokens"][:2].to(DEVICE))
                torch.cuda.synchronize()
            print("CUDA ResNet18 smoke test passed.")
        except RuntimeError as exc:
            if looks_like_cuda_runtime_error(exc):
                print("CUDA model smoke test failed, rebuilding model on CPU.")
                print("RuntimeError:", str(exc)[:500])
                DEVICE = torch.device("cpu")
                model = ImageTextContrastiveModel(len(TOKENIZER.stoi), CFG.EMBED_DIM).to(DEVICE)
            else:
                raise
    return model

model = build_model_with_cuda_smoke_test()
print("Device after model smoke test:", DEVICE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 8. Symmetric InfoNCE Loss

The diagonal of the batch similarity matrix contains matching image-caption pairs. The loss averages image-to-text and text-to-image cross entropy.

In [ ]:
def symmetric_infonce_loss(image_emb, text_emb, temperature=0.07):
    logits = image_emb @ text_emb.T / temperature
    labels = torch.arange(image_emb.size(0), device=image_emb.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2.0

with torch.no_grad():
    images = batch["image"].to(DEVICE)
    tokens = batch["tokens"].to(DEVICE)
    image_emb, text_emb = model(images, tokens)
    test_loss = symmetric_infonce_loss(image_emb, text_emb, CFG.TEMPERATURE)
print("Sanity loss:", float(test_loss.detach().cpu()))

## 9. Training From Scratch

This trains for at least one epoch on Flickr8k image-caption pairs and saves both the epoch-1 checkpoint and training history.

In [ ]:
def move_batch(batch, device):
    return {
        "image": batch["image"].to(device, non_blocking=True),
        "tokens": batch["tokens"].to(device, non_blocking=True),
        "caption": batch["caption"],
        "image_id": batch["image_id"],
    }


def train_one_epoch(model, loader, optimizer, temperature, device, max_batches=None):
    model.train()
    running_loss = 0.0
    n_batches = 0
    t0 = time.time()
    for step, batch in enumerate(loader, start=1):
        if max_batches is not None and step > max_batches:
            break
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)
        image_emb, text_emb = model(batch["image"], batch["tokens"])
        loss = symmetric_infonce_loss(image_emb, text_emb, temperature)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += float(loss.item())
        n_batches += 1
        if step % 100 == 0:
            print(f"  batch {step:04d}/{len(loader)} | loss={running_loss / n_batches:.4f}")
    return running_loss / max(n_batches, 1), round(time.time() - t0, 2), n_batches

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
history = []

print("Training: Simplified ImageBind-style image-text contrastive learning on Flickr8k")
for epoch in range(1, CFG.EPOCHS + 1):
    train_loss, seconds, batches_seen = train_one_epoch(model, train_loader, optimizer, CFG.TEMPERATURE, DEVICE)
    row = {"epoch": epoch, "train_loss": train_loss, "seconds": seconds, "batches": batches_seen, "temperature": CFG.TEMPERATURE}
    history.append(row)
    print(f"Epoch {epoch}/{CFG.EPOCHS} | train_loss={train_loss:.4f} | batches={batches_seen} | time={seconds}s")

ckpt_path = CHECKPOINT_DIR / "flickr8k_image_text_epoch1.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": asdict(CFG),
    "tokenizer_stoi": TOKENIZER.stoi,
    "epoch": CFG.EPOCHS,
    "history": history,
}, ckpt_path)

history_path = OUTPUT_DIR / "flickr8k_train_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print("Saved checkpoint:", ckpt_path)
print("Saved history:", history_path)

## 10. Retrieval Evaluation

Evaluation extracts embeddings for the test split. Because each image has multiple captions, image-to-text retrieval treats any caption with the same `image_id` as a positive. Text-to-image retrieval treats the matching image filename as the positive.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, device):
    model.eval()
    image_chunks, text_chunks = [], []
    image_ids, captions = [], []
    for batch in loader:
        batch_dev = move_batch(batch, device)
        image_emb, text_emb = model(batch_dev["image"], batch_dev["tokens"])
        image_chunks.append(image_emb.cpu())
        text_chunks.append(text_emb.cpu())
        image_ids.extend(batch["image_id"])
        captions.extend(batch["caption"])
    return {
        "image_emb_rows": torch.cat(image_chunks, dim=0),
        "text_emb": torch.cat(text_chunks, dim=0),
        "image_ids": image_ids,
        "captions": captions,
    }


def unique_image_embeddings(row_image_emb, image_ids):
    first_index = {}
    unique_ids = []
    for idx, image_id in enumerate(image_ids):
        if image_id not in first_index:
            first_index[image_id] = idx
            unique_ids.append(image_id)
    indices = torch.tensor([first_index[x] for x in unique_ids], dtype=torch.long)
    return row_image_emb[indices], unique_ids


def recall_at_k_grouped(similarity, positive_sets, ks=(1, 5, 10)):
    recalls = {}
    max_k = min(max(ks), similarity.size(1))
    ranked = similarity.topk(max_k, dim=1).indices.cpu().tolist()
    for k in ks:
        k_eff = min(k, similarity.size(1))
        hits = []
        for row_idx, positives in enumerate(positive_sets):
            topk = set(ranked[row_idx][:k_eff])
            hits.append(len(topk.intersection(positives)) > 0)
        recalls[f"R@{k}"] = float(np.mean(hits)) if hits else 0.0
    return recalls


def compute_retrieval_metrics(extracted):
    text_emb = extracted["text_emb"]
    image_emb_unique, unique_ids = unique_image_embeddings(extracted["image_emb_rows"], extracted["image_ids"])
    image_to_index = {image_id: idx for idx, image_id in enumerate(unique_ids)}

    caption_indices_by_image = defaultdict(set)
    for idx, image_id in enumerate(extracted["image_ids"]):
        caption_indices_by_image[image_id].add(idx)

    sim_i2t = image_emb_unique @ text_emb.T
    i2t_positive_sets = [caption_indices_by_image[image_id] for image_id in unique_ids]
    i2t = recall_at_k_grouped(sim_i2t, i2t_positive_sets, ks=(1, 5, 10))

    sim_t2i = text_emb @ image_emb_unique.T
    t2i_positive_sets = [{image_to_index[image_id]} for image_id in extracted["image_ids"]]
    t2i = recall_at_k_grouped(sim_t2i, t2i_positive_sets, ks=(1, 5, 10))

    return {
        "image_to_text": i2t,
        "text_to_image": t2i,
        "num_test_caption_pairs": len(extracted["captions"]),
        "num_test_images": len(unique_ids),
    }, image_emb_unique, unique_ids

extracted = extract_embeddings(model, test_loader, DEVICE)
eval_results, test_image_emb_unique, test_unique_ids = compute_retrieval_metrics(extracted)
eval_results["dataset"] = "Flickr8k"
eval_results["model"] = "Simplified ImageBind-style image-text contrastive learning on Flickr8k"
eval_results["epochs"] = CFG.EPOCHS
eval_results["temperature"] = CFG.TEMPERATURE

eval_path = OUTPUT_DIR / "flickr8k_eval_results.json"
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2)
print(json.dumps(eval_results, indent=2))
print("Saved evaluation:", eval_path)

## 11. Visualization

This section saves a learning curve and five retrieval examples from the test set. Each example shows the query image and top retrieved captions.

In [ ]:
def plot_learning_curve(history, output_path):
    epochs = [h["epoch"] for h in history]
    losses = [h["train_loss"] for h in history]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(epochs, losses, marker="o")
    ax.set_title("Flickr8k Image-Text Contrastive Training")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train loss")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()


def save_retrieval_examples(extracted, image_emb_unique, unique_ids, image_dir, output_path, n_examples=5, top_k=3):
    text_emb = extracted["text_emb"]
    sim = image_emb_unique @ text_emb.T
    id_to_path = {row.image_id: row.image_path for row in test_df.drop_duplicates("image_id").itertuples()}
    n_examples = min(n_examples, len(unique_ids))
    chosen = list(range(n_examples))

    fig, axes = plt.subplots(n_examples, 2, figsize=(12, 3.0 * n_examples))
    if n_examples == 1:
        axes = np.array([axes])

    for row_pos, image_idx in enumerate(chosen):
        image_id = unique_ids[image_idx]
        pil_img = Image.open(id_to_path[image_id]).convert("RGB")
        axes[row_pos, 0].imshow(pil_img)
        axes[row_pos, 0].set_title(image_id)
        axes[row_pos, 0].axis("off")

        top_idx = sim[image_idx].topk(min(top_k, sim.size(1))).indices.cpu().tolist()
        lines = []
        for rank, cap_idx in enumerate(top_idx, start=1):
            cap = extracted["captions"][cap_idx]
            cap_image = extracted["image_ids"][cap_idx]
            marker = "*" if cap_image == image_id else " "
            lines.append(f"{rank}. [{marker}] {cap}")
        text = "\n\n".join(textwrap.fill(x, width=80) for x in lines)
        axes[row_pos, 1].text(0.0, 0.95, text, va="top", fontsize=10)
        axes[row_pos, 1].axis("off")

    fig.suptitle("Image-to-Text Retrieval Examples", fontsize=14)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()

learning_curve_path = OUTPUT_DIR / "flickr8k_learning_curve.png"
retrieval_examples_path = OUTPUT_DIR / "flickr8k_retrieval_examples.png"
plot_learning_curve(history, learning_curve_path)
save_retrieval_examples(extracted, test_image_emb_unique, test_unique_ids, IMAGES_DIR, retrieval_examples_path)
print("Saved learning curve:", learning_curve_path)
print("Saved retrieval examples:", retrieval_examples_path)

## 12. Optional Mini Ablation: Temperature

This optional cell compares `TEMPERATURE = [0.05, 0.07, 0.1]`. Each setting trains from scratch for one quick epoch capped at 100 batches, then evaluates retrieval on the test split.

In [ ]:
RUN_TEMPERATURE_ABLATION = True
TEMPERATURE_VALUES = [0.05, 0.07, 0.1]


def run_temperature_ablation():
    results = []
    for temp in TEMPERATURE_VALUES:
        print(f"\nAblation temperature={temp}")
        ab_model = ImageTextContrastiveModel(len(TOKENIZER.stoi), CFG.EMBED_DIM).to(DEVICE)
        ab_optimizer = torch.optim.AdamW(ab_model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
        loss, seconds, batches_seen = train_one_epoch(
            ab_model,
            train_loader,
            ab_optimizer,
            temperature=temp,
            device=DEVICE,
            max_batches=CFG.ABLATION_MAX_BATCHES,
        )
        ab_extracted = extract_embeddings(ab_model, test_loader, DEVICE)
        ab_eval, _, _ = compute_retrieval_metrics(ab_extracted)
        row = {
            "temperature": temp,
            "train_loss": loss,
            "seconds": seconds,
            "batches": batches_seen,
            "image_to_text": ab_eval["image_to_text"],
            "text_to_image": ab_eval["text_to_image"],
        }
        results.append(row)
        print(row)
    return results

if RUN_TEMPERATURE_ABLATION:
    ablation_results = run_temperature_ablation()
else:
    ablation_results = []

ablation_path = OUTPUT_DIR / "flickr8k_ablation_temperature.json"
with open(ablation_path, "w") as f:
    json.dump(ablation_results, f, indent=2)
print("Saved temperature ablation:", ablation_path)

## 13. Final Summary

The final table reports the dataset, modality, pair type, epochs, final loss, and both retrieval directions.

In [ ]:
final_loss = history[-1]["train_loss"] if history else None
summary = pd.DataFrame([
    {
        "Dataset": "Flickr8k",
        "Modality": "Image + Text",
        "Pair type": "image-caption positive pairs",
        "Epochs": CFG.EPOCHS,
        "Loss": final_loss,
        "I2T R@1": eval_results["image_to_text"]["R@1"],
        "I2T R@5": eval_results["image_to_text"]["R@5"],
        "I2T R@10": eval_results["image_to_text"]["R@10"],
        "T2I R@1": eval_results["text_to_image"]["R@1"],
        "T2I R@5": eval_results["text_to_image"]["R@5"],
        "T2I R@10": eval_results["text_to_image"]["R@10"],
    }
])
print("Simplified ImageBind-style image-text contrastive learning on Flickr8k.")
display(summary)

summary_path = OUTPUT_DIR / "flickr8k_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary.to_dict(orient="records"), f, indent=2)
print("Saved summary:", summary_path)